In [53]:
from sklearn.metrics import mean_absolute_error, r2_score

from pycaret.regression import *
import lightgbm as lgb
import mlflow

import pandas as pd

In [54]:
print(mlflow.__version__)

2.0.0


In [55]:
# client = mlflow.MlflowClient()     
# registered_models = client.search_registered_models()

# for model in registered_models:
#     if model.name.startswith("hw_"):
#         client.delete_registered_model(model.name)

In [76]:
mlflow.set_tracking_uri("http://ec2-54-160-110-158.compute-1.amazonaws.com:5000/")

In [ ]:
registered_models = client.search_registered_models()

hw_models = {}
hw_models['all_crop'] = {}
hw_models['top_crop'] = {}
for model in registered_models:
    if model.name.startswith("hw_") and 'all_crop' in model.name:
        name = model.name.replace("hw_", "").replace("_all_crop_ts", "")
        hw_models['all_crop'][name] = mlflow.pyfunc.load_model(f"models:/{model.name}/Production")

    if model.name.startswith("hw_") and 'top_crop' in model.name:
        name = model.name.replace("hw_", "").replace("_top_crop_ts", "")
        hw_models['top_crop'][name] = mlflow.pyfunc.load_model(f"models:/{model.name}/Production")

init_year = 2000


In [ ]:
def get_base_data(input: PredictionInputHW):
    country = input.country
    crop_type = input.crop_type
    base_data = {}
    if crop_type == 'all_crop':
        model = hw_models['all_crop'][country]
        m_data = model.unwrap_python_model().model.data.endog
        base_data = {k: v for k, v in zip([init_year+i for i in range(len(m_data))], m_data)}
        return base_data
    elif crop_type == 'top_crop':
        model = hw_models['top_crop'][country]
        base_data = model.unwrap_python_model().model.data.endog
        return base_data
    else:
        return base_data

def predict_model_hw(input: PredictionInputHW):
    country = input.country
    crop_type = input.crop_type
    
    if crop_type == 'all_crop':
        model = hw_models['all_crop'][country]
    else:
        model = hw_models['top_crop'][country]

    prediction = model.predict(input.data)
    return prediction

In [91]:
base_data = {}

model = hw_models['all_crop']['austria']
m_data = model.unwrap_python_model().model.data.endog
base_data = {k: v for k, v in zip([init_year+i for i in range(len(m_data))], m_data)}
    

In [106]:
years = 2023

m_data = model.unwrap_python_model().model.data.endog
max_base = max([init_year+i for i in range(len(m_data))]) + 1
pred_years = [i for i in range(max_base, years+1)]

base_data = {k: v for k, v in zip(pred_years, model.predict(pred_years))}
    


In [109]:
pred_years

[2022, 2023]

In [111]:
list(model.predict(pred_years))

[26832.82103006469, 26560.171216409264]

In [104]:
pred_years = [i for i in range(max_base, years+1)]
pred_years

[2022, 2023]

In [101]:
max_base

2022

In [ ]:
model.predict(input.data)

{2000: 18222.299875818193,
 2001: 19001.240032278,
 2002: 19081.52985738218,
 2003: 16885.50994320959,
 2004: 20643.739856943488,
 2005: 19902.34005060047,
 2006: 18292.130238115788,
 2007: 19048.050015367568,
 2008: 22273.33016501367,
 2009: 20592.909884944558,
 2010: 37948.81969670206,
 2011: 42547.83998894133,
 2012: 38040.12039741501,
 2013: 37989.55014877394,
 2014: 44390.62035212666,
 2015: 27391.969961268827,
 2016: 32018.499898251146,
 2017: 27195.19001009874,
 2018: 26167.239942926913,
 2019: 28125.970355857164,
 2020: 30174.620158271864,
 2021: 29284.940003326163}

In [86]:
model.unwrap_python_model()

In [68]:
logged_model_hw.predict([0])

array([26832.82103006])

In [71]:
m1.model.fittedvalues

array([13383.15419728, 19091.17876265, 16084.04712035, 18158.18706822,
       19120.33864496, 19644.47060777, 21765.51744017, 16109.93773647,
       18137.17208113, 23333.94854353, 20519.5067402 , 36082.96765478,
       38208.26656653, 37803.86410218, 39907.29759285, 42750.51576324,
       32644.6708057 , 29218.03256359, 27361.70317013, 28385.10670727,
       27513.67270129, 31515.98998968])

In [75]:
m1.model.data.endog

array([18222.29987582, 19001.24003228, 19081.52985738, 16885.50994321,
       20643.73985694, 19902.3400506 , 18292.13023812, 19048.05001537,
       22273.33016501, 20592.90988494, 37948.8196967 , 42547.83998894,
       38040.12039742, 37989.55014877, 44390.62035213, 27391.96996127,
       32018.49989825, 27195.1900101 , 26167.23994293, 28125.97035586,
       30174.62015827, 29284.94000333])

In [89]:
init_year = 2000